In [7]:
# ── CELL 1: Imports & Configuration ───────────────────────────────────────
import pandas as pd
import numpy as np
import pickle
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from pathlib import Path

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Paths
RAW_PATH = Path("voraus-ad-dataset-100hz.parquet")
OUTPUT_DIR = Path("data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load data
print("Loading data...")
df = pd.read_parquet(RAW_PATH)
print(f"Raw data shape: {df.shape}")

# ============================================================================
# Step 1.1: Remove metadata columns (keep only sensors)
# ============================================================================
METADATA_COLS = ['time', 'sample', 'anomaly', 'category', 'setting', 'action', 'active']

# All remaining columns are sensors
all_sensors = [c for c in df.columns if c not in METADATA_COLS]
print(f"\nTotal sensor columns: {len(all_sensors)}")

# Remove constant columns (no information)
constant_sensors = [col for col in all_sensors if df[col].nunique() <= 1]
sensors_to_keep = [col for col in all_sensors if col not in constant_sensors]

print(f"Removed {len(constant_sensors)} constant sensors")
print(f"Keeping {len(sensors_to_keep)} sensors")

Loading data...
Raw data shape: (2321690, 137)

Total sensor columns: 130
Removed 1 constant sensors
Keeping 129 sensors


In [8]:
# ── CELL 2: Find Minimum Sequence Length ───────────────────────────────────
# We need all sequences to have the same length for batch training

# Calculate length of each sample
sample_lengths = df.groupby('sample').size()
print(f"Sample length statistics:")
print(f"  Min: {sample_lengths.min()}")
print(f"  Max: {sample_lengths.max()}")
print(f"  Mean: {sample_lengths.mean():.1f}")
print(f"  Median: {sample_lengths.median():.1f}")

# Choose sequence length (use minimum to avoid padding)
# Or use a percentile to avoid losing too many samples
SEQ_LEN = sample_lengths.min()  # 1028 from your original notebook
print(f"\nUsing sequence length: {SEQ_LEN}")

# Check how many samples would be truncated if we use this length
samples_too_short = (sample_lengths < SEQ_LEN).sum()
print(f"Samples shorter than {SEQ_LEN}: {samples_too_short}")

Sample length statistics:
  Min: 986
  Max: 1164
  Mean: 1094.1
  Median: 1096.0

Using sequence length: 986
Samples shorter than 986: 0


In [9]:
# ── CELL 3: Split Data by Sample (No Leakage!) ────────────────────────────
# Create metadata DataFrame (one row per sample)
sample_meta = df.groupby('sample').agg(
    anomaly=('anomaly', 'first'),
    n_timesteps=('time', 'count')
).reset_index()

print(f"Total samples: {len(sample_meta)}")
print(f"Normal samples: {(~sample_meta['anomaly']).sum()}")
print(f"Anomaly samples: {sample_meta['anomaly'].sum()}")

# Separate normal and anomaly samples
normal_samples = sample_meta[~sample_meta['anomaly']]
anomaly_samples = sample_meta[sample_meta['anomaly']]

# Split normal samples (70% train, 15% val, 15% test)
train_norm, temp_norm = train_test_split(
    normal_samples, 
    test_size=0.30, 
    random_state=42
)
val_norm, test_norm = train_test_split(
    temp_norm, 
    test_size=0.50, 
    random_state=42
)

# Split anomaly samples (50% val, 50% test)
val_anom, test_anom = train_test_split(
    anomaly_samples, 
    test_size=0.50, 
    random_state=42
)

# Combine
train_samples = train_norm  # ONLY normal for training (unsupervised)
val_samples = pd.concat([val_norm, val_anom])  # Both for validation
test_samples = pd.concat([test_norm, test_anom])  # Both for testing

print(f"\nSplit summary:")
print(f"  Train: {len(train_samples)} samples (all normal)")
print(f"  Validation: {len(val_samples)} samples ({len(val_norm)} normal, {len(val_anom)} anomaly)")
print(f"  Test: {len(test_samples)} samples ({len(test_norm)} normal, {len(test_anom)} anomaly)")

Total samples: 2122
Normal samples: 1367
Anomaly samples: 755

Split summary:
  Train: 956 samples (all normal)
  Validation: 582 samples (205 normal, 377 anomaly)
  Test: 584 samples (206 normal, 378 anomaly)


In [10]:
# ── CELL 4: Load Sequences (Keep 3D Shape) ────────────────────────────────
def load_sequence(sample_id, df, sensors, seq_len):
    """
    Load one sample's sensor data.
    Returns shape: (seq_len, n_sensors)
    """
    # Get data for this sample
    sample_data = df[df['sample'] == sample_id][sensors].values
    
    # Take last seq_len timesteps (or truncate from end)
    if len(sample_data) >= seq_len:
        sequence = sample_data[-seq_len:]
    else:
        # Pad with zeros if shorter (shouldn't happen if we use min length)
        pad_len = seq_len - len(sample_data)
        sequence = np.vstack([np.zeros((pad_len, len(sensors))), sample_data])
    
    return sequence

def build_sequences(sample_ids, df, sensors, seq_len):
    """
    Stack sequences into (N, seq_len, n_sensors) array.
    """
    sequences = []
    for sid in sample_ids:
        seq = load_sequence(sid, df, sensors, seq_len)
        sequences.append(seq)
    return np.array(sequences)

print("Building sequences (this may take a minute)...")

# Build sequences for each split
X_train_raw = build_sequences(train_samples['sample'], df, sensors_to_keep, SEQ_LEN)
X_val_raw = build_sequences(val_samples['sample'], df, sensors_to_keep, SEQ_LEN)
X_test_raw = build_sequences(test_samples['sample'], df, sensors_to_keep, SEQ_LEN)

# Labels for validation and test
y_val = val_samples['anomaly'].astype(int).values
y_test = test_samples['anomaly'].astype(int).values

print(f"\nSequence shapes:")
print(f"  X_train: {X_train_raw.shape}")  # (N_train, 1028, n_sensors)
print(f"  X_val: {X_val_raw.shape}")      # (N_val, 1028, n_sensors)
print(f"  X_test: {X_test_raw.shape}")    # (N_test, 1028, n_sensors)
print(f"\nAnomaly rate in test: {y_test.mean():.1%}")

Building sequences (this may take a minute)...

Sequence shapes:
  X_train: (956, 986, 129)
  X_val: (582, 986, 129)
  X_test: (584, 986, 129)

Anomaly rate in test: 64.7%


In [11]:
# ── CELL 5: Scale the Data (IMPORTANT: Scale per feature, not per timestep)
print("Scaling data...")

# Reshape to (N*T, F) for fitting scaler
n_train, T, F = X_train_raw.shape
X_train_flat = X_train_raw.reshape(-1, F)

# Fit scaler on training data only
scaler = StandardScaler()
scaler.fit(X_train_flat)

# Transform all data
X_train = scaler.transform(X_train_flat).reshape(n_train, T, F).astype('float32')

# For validation and test
n_val, T, F = X_val_raw.shape
X_val = scaler.transform(X_val_raw.reshape(-1, F)).reshape(n_val, T, F).astype('float32')

n_test, T, F = X_test_raw.shape
X_test = scaler.transform(X_test_raw.reshape(-1, F)).reshape(n_test, T, F).astype('float32')

print(f"Data scaled to {X_train.dtype}")
print(f"X_train shape after scaling: {X_train.shape}")
print(f"Mean after scaling: {X_train.mean():.4f}")
print(f"Std after scaling: {X_train.std():.4f}")

Scaling data...
Data scaled to float32
X_train shape after scaling: (956, 986, 129)
Mean after scaling: 0.0000
Std after scaling: 1.0000


In [12]:
# ── CELL 6: Save Processed Data ───────────────────────────────────────────
# Save as numpy arrays (easier to load later)
np.savez_compressed(
    OUTPUT_DIR / "sequences.npz",
    X_train=X_train,
    X_val=X_val,
    X_test=X_test,
    y_val=y_val,
    y_test=y_test
)

# Save scaler
with open(OUTPUT_DIR / "scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

# Save metadata
train_samples.to_parquet(OUTPUT_DIR / "train_metadata.parquet")
val_samples.to_parquet(OUTPUT_DIR / "val_metadata.parquet")
test_samples.to_parquet(OUTPUT_DIR / "test_metadata.parquet")

# Save configuration
config = {
    'seq_len': SEQ_LEN,
    'n_features': F,
    'n_train_samples': len(train_samples),
    'n_val_samples': len(val_samples),
    'n_test_samples': len(test_samples),
    'sensors': sensors_to_keep
}

with open(OUTPUT_DIR / "config.pkl", "wb") as f:
    pickle.dump(config, f)

print(f"\n✓ Saved to {OUTPUT_DIR}/")
print(f"  - sequences.npz (X_train, X_val, X_test, y_val, y_test)")
print(f"  - scaler.pkl")
print(f"  - *_metadata.parquet")
print(f"  - config.pkl")


✓ Saved to data/processed/
  - sequences.npz (X_train, X_val, X_test, y_val, y_test)
  - scaler.pkl
  - *_metadata.parquet
  - config.pkl


In [13]:
# ── CELL 7: Verification ──────────────────────────────────────────────────
print("="*50)
print("VERIFICATION")
print("="*50)

# Reload and verify
data = np.load(OUTPUT_DIR / "sequences.npz")
X_train_loaded = data['X_train']
X_val_loaded = data['X_val']
X_test_loaded = data['X_test']
y_val_loaded = data['y_val']
y_test_loaded = data['y_test']

with open(OUTPUT_DIR / "config.pkl", "rb") as f:
    config_loaded = pickle.load(f)

# Check shapes
assert X_train_loaded.shape == X_train.shape
assert X_val_loaded.shape == X_val.shape
assert X_test_loaded.shape == X_test.shape

# Check for NaNs
assert not np.isnan(X_train_loaded).any(), "NaNs found in training data"

print(f"✓ Training sequences: {X_train_loaded.shape}")
print(f"✓ Validation sequences: {X_val_loaded.shape} ({y_val_loaded.sum()} anomalies)")
print(f"✓ Test sequences: {X_test_loaded.shape} ({y_test_loaded.sum()} anomalies)")
print(f"✓ Sequence length: {config_loaded['seq_len']} timesteps")
print(f"✓ Features per timestep: {config_loaded['n_features']}")
print(f"✓ Data type: {X_train_loaded.dtype}")
print(f"✓ No NaN values found")

print("\n✅ Data preparation complete! Ready for LSTM autoencoder.")

VERIFICATION
✓ Training sequences: (956, 986, 129)
✓ Validation sequences: (582, 986, 129) (377 anomalies)
✓ Test sequences: (584, 986, 129) (378 anomalies)
✓ Sequence length: 986 timesteps
✓ Features per timestep: 129
✓ Data type: float32
✓ No NaN values found

✅ Data preparation complete! Ready for LSTM autoencoder.


In [14]:
# ── CHECK IF YOUR HARDWARE CAN HANDLE FULL LENGTH ─────────────────────────
import psutil
import torch

# Calculate memory requirements
SEQ_LEN = 986
N_FEATURES = 129
N_SAMPLES = 956

# Memory for training data (float32)
memory_gb = (N_SAMPLES * SEQ_LEN * N_FEATURES * 4) / (1024**3)
print(f"Training data memory: {memory_gb:.2f} GB")

# Check available RAM
available_ram = psutil.virtual_memory().available / (1024**3)
print(f"Available RAM: {available_ram:.2f} GB")

# Check for GPU
if torch.cuda.is_available():
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU memory: {gpu_memory:.2f} GB")
    print(f"Can fit on GPU: {memory_gb < gpu_memory * 0.8}")
else:
    print("No GPU detected - will use CPU (much slower)")

if memory_gb > available_ram * 0.5:
    print("\n⚠️ Warning: Full length may cause memory issues")
    print("   Consider using windows or reduced length")
else:
    print("\n✓ Your system can handle full length sequences!")

Training data memory: 0.45 GB
Available RAM: 17.93 GB
No GPU detected - will use CPU (much slower)

✓ Your system can handle full length sequences!


In [15]:
# ── CHECK MPS AVAILABILITY ────────────────────────────────────────────────
import torch

# Check MPS (Apple Silicon GPU)
mps_available = torch.backends.mps.is_available()
mps_built = torch.backends.mps.is_built()

print(f"MPS available: {mps_available}")
print(f"MPS built: {mps_built}")

# Set device
if mps_available:
    device = torch.device("mps")
    print("✅ Using MPS (Apple Silicon GPU) - Much faster than CPU!")
else:
    device = torch.device("cpu")
    print("⚠️ MPS not available, using CPU")

# Test with a small tensor
if mps_available:
    test_tensor = torch.ones(10, 10).to(device)
    print(f"Test tensor on MPS: {test_tensor.device}")

MPS available: True
MPS built: True
✅ Using MPS (Apple Silicon GPU) - Much faster than CPU!
Test tensor on MPS: mps:0
